In [ ]:
# Cell 1 — Imports  (pyarrow MUST be first: Windows DLL fix)
import pyarrow
import transformers
import torch
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm

print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# Cell 2 — Config, load model, retrain probe at layer 9 explicitly
from src.utils.model_loader import get_model
from src.probing.faithfulness_detector import FaithfulnessProbe, generate_faithfulness_labels
from src.probing.cot_probe import extract_cot_activations
from data.dataset_loader import load_gsm8k
import numpy as np

MODEL_NAME = 'gpt2'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR    = '../outputs'
ALPHA      = 20.0
MAX_TOKENS = 30
N_TEST     = 5
NLDD_THRESHOLD = 0.1

PROBE_LAYER_OVERRIDE = 9  # override automatic best-layer selection — see notebook 02
                           # analysis for justification: layer 0 and layer 9 tie within
                           # 0.002 CV accuracy (noise level); layer 9 is preferred as it
                           # lies in the upper semantic processing layers and yields
                           # cosine similarity 0.67 between probe direction and
                           # mean-difference steering vector.

print(f'Loading {MODEL_NAME} ...')
model = get_model(MODEL_NAME, device=DEVICE)
print(f'Loaded  layers={model.cfg.n_layers}  d_model={model.cfg.d_model}')

# Retrain probe at PROBE_LAYER_OVERRIDE on 100 GSM8K samples.
# probe_best.pkl on disk was saved at layer 0 — we bypass it here.
print(f'\nRetraining probe at layer {PROBE_LAYER_OVERRIDE} on 100 GSM8K samples...')
train_samples = load_gsm8k(split='test', max_samples=100)
per_step_acts_l9 = []
per_step_labels_l9 = []

for sample in tqdm(train_samples, desc=f'Layer-{PROBE_LAYER_OVERRIDE} probe set'):
    question, steps = sample['question'], sample['steps']
    if len(steps) < 2:
        continue
    prompt = 'Q: ' + question + ' A: ' + ' '.join(steps)
    try:
        label_tuples = generate_faithfulness_labels(
            model, prompt, steps,
            strategy='number_perturbation', nldd_threshold=NLDD_THRESHOLD)
    except Exception:
        continue
    acts = extract_cot_activations(model, prompt)
    for _, is_faithful, _ in label_tuples:
        per_step_acts_l9.append(acts[PROBE_LAYER_OVERRIDE].mean(dim=0).numpy())
        per_step_labels_l9.append(is_faithful)

acts_arr = np.array(per_step_acts_l9)
probe = FaithfulnessProbe(layer=PROBE_LAYER_OVERRIDE)
cv_acc = probe.train(acts_arr, per_step_labels_l9)
print(f'Probe layer={PROBE_LAYER_OVERRIDE}  CV accuracy={cv_acc:.3f}  '
      f'n_train={len(per_step_labels_l9)}  '
      f'faithful_fraction={sum(per_step_labels_l9)/max(1,len(per_step_labels_l9)):.1%}')

# Load probe metadata from notebook 02 for reference
meta_path = os.path.join(OUT_DIR, 'probe_meta.json')
if os.path.exists(meta_path):
    with open(meta_path) as f:
        probe_meta = json.load(f)
    print('Probe meta (nb02 reference):', probe_meta)
else:
    probe_meta = {}

In [ ]:
# Cell 3 — Compute steering vectors
# Method 1: Use probe.direction (linear probe weight vector)
# Method 2: compute_steering_vector_from_dicts (mean-difference over activation dicts)
from src.probing.cot_probe import extract_cot_activations
from src.probing.faithfulness_detector import generate_faithfulness_labels, corrupt_step
from src.steering.steering_vectors import compute_steering_vector_from_dicts, save_steering_vector
from data.dataset_loader import load_gsm8k

steering_layer = probe.layer if probe else (model.cfg.n_layers // 2)

# Method 1: probe direction as steering vector
if probe is not None:
    v_probe = probe.direction.to(DEVICE)
    print(f'Method 1 steering vector (probe direction)  shape={v_probe.shape}  norm={v_probe.norm():.4f}')
else:
    v_probe = None
    print('Method 1: no probe available.')

# Method 2: mean-difference over a small set of faithful/unfaithful activations
calib_samples    = load_gsm8k(split='test', max_samples=10)
faithful_dicts   = []
unfaithful_dicts = []

for sample in tqdm(calib_samples, desc='Building steering set'):
    question, steps = sample['question'], sample['steps']
    if len(steps) < 2:
        continue
    prompt = 'Q: ' + question + ' A: ' + ' '.join(steps)
    try:
        labels = generate_faithfulness_labels(model, prompt, steps, nldd_threshold=0.1)
    except Exception:
        continue
    acts = extract_cot_activations(model, prompt)
    for idx, is_faithful, _ in labels:
        (faithful_dicts if is_faithful else unfaithful_dicts).append(acts)

print(f'Faithful acts: {len(faithful_dicts)}  |  Unfaithful acts: {len(unfaithful_dicts)}')

if len(faithful_dicts) >= 2 and len(unfaithful_dicts) >= 2:
    v_meandiff = compute_steering_vector_from_dicts(faithful_dicts, unfaithful_dicts, layer=steering_layer)
    v_meandiff = v_meandiff.to(DEVICE)
    save_steering_vector(v_meandiff, os.path.join(OUT_DIR, 'steering_vec_meandiff.pt'),
                         metadata={'layer': steering_layer, 'method': 'mean_difference', 'model': MODEL_NAME})
    print(f'Method 2 steering vector (mean-diff)  shape={v_meandiff.shape}  norm={v_meandiff.norm():.4f}')
    # Cosine similarity between the two methods
    if v_probe is not None:
        import torch.nn.functional as F
        sim = float(F.cosine_similarity(v_probe.unsqueeze(0), v_meandiff.unsqueeze(0)))
        print(f'Cosine similarity (probe vs mean-diff): {sim:.4f}')
    steering_vector = v_meandiff
else:
    print('Insufficient calibration data for Method 2; falling back to Method 1.')
    steering_vector = v_probe

print(f'Using steering vector at layer {steering_layer}')

In [ ]:
# Cell 4 — Baseline run_pipeline on 5 test samples (no steering)
from src.pipeline.closed_loop import run_pipeline
from data.dataset_loader import load_gsm8k

test_samples = load_gsm8k(split='test', max_samples=50)  # larger pool; take first N_TEST valid
test_prompts = []
for s in test_samples:
    if len(test_prompts) >= N_TEST:
        break
    if len(s['steps']) >= 2:
        test_prompts.append('Q: ' + s['question'] + ' A: ')

print(f'Running BASELINE pipeline on {len(test_prompts)} prompts (no steering)...')
baseline_traces = []
for i, prompt in enumerate(tqdm(test_prompts, desc='Baseline')):
    trace = run_pipeline(model, prompt, probe=probe, max_tokens=MAX_TOKENS)
    baseline_traces.append(trace)
    if probe is not None and trace.confidences:
        mean_conf = float(np.mean(trace.confidences))
        print(f'  Sample {i}: mean confidence={mean_conf:.3f}  tokens={len(trace.generated_tokens)}')

print('Baseline complete.')

In [ ]:
# Cell 5 — Steered run_pipeline (alpha=1.0) on same prompts
print(f'Running STEERED pipeline on {len(test_prompts)} prompts (alpha={ALPHA})...')
steered_traces = []
for i, prompt in enumerate(tqdm(test_prompts, desc='Steered')):
    trace = run_pipeline(
        model, prompt,
        probe=probe,
        steering_vector=steering_vector,
        layer=steering_layer,
        max_tokens=MAX_TOKENS,
        threshold=0.5,
        alpha=ALPHA,
    )
    steered_traces.append(trace)
    n_steer = len(trace.steering_event_indices)
    print(f'  Sample {i}: steering_events={n_steer}  tokens={len(trace.generated_tokens)}')

print('Steered complete.')

In [ ]:
# Cell 6 — Figure 3: 3x2 faithfulness trace grid, red lines at steering events, stats table
from src.utils.visualizer import plot_faithfulness_trace

n_rows, n_cols = 2, 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 7))

for idx in range(n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    ax = axes[row][col]
    if idx >= len(test_prompts):
        ax.axis('off')
        continue
    bt = baseline_traces[idx]
    st = steered_traces[idx]
    if not bt.confidences:
        ax.text(0.5, 0.5, 'No probe data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Sample {idx}')
        continue
    steps = range(len(bt.confidences))
    ax.plot(steps, bt.confidences, color='steelblue', lw=1.5, label='Baseline', alpha=0.8)
    if st.confidences:
        ax.plot(range(len(st.confidences)), st.confidences,
                color='mediumseagreen', lw=1.5, label='Steered', alpha=0.8)
    for si in st.steering_event_indices:
        ax.axvline(si, color='crimson', ls='--', lw=1, alpha=0.8,
                   label='Steer' if si == st.steering_event_indices[0] else None)
    ax.set_ylim(0, 1)
    ax.set_title(f'Sample {idx} ({len(st.steering_event_indices)} events)', fontsize=9)
    ax.set_xlabel('Token step', fontsize=8)
    ax.set_ylabel('Confidence', fontsize=8)
    if idx == 0:
        ax.legend(fontsize=7)

fig.suptitle('Figure 3: Faithfulness Traces — Baseline vs Steered', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'figure3_steering_traces.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure3_steering_traces.png')

# Stats table
rows = []
for i, (bt, st) in enumerate(zip(baseline_traces, steered_traces)):
    rows.append({
        'Sample': i,
        'Baseline mean conf': f'{np.mean(bt.confidences):.3f}' if bt.confidences else 'n/a',
        'Steered mean conf':  f'{np.mean(st.confidences):.3f}' if st.confidences else 'n/a',
        'Steering events': len(st.steering_event_indices),
        'Tokens generated': len(bt.generated_tokens),
    })
stats_df = pd.DataFrame(rows)
print(stats_df.to_string(index=False))

In [ ]:
# Cell 7
print('Notebook 03 complete')